# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [2]:
# !pip install -qU datasets pinecone-client==3.1.0 sentence-transformers torch
import torch
import pinecone
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

print("Environment ready.")

c:\Users\profe\Documents\Ironhack\lab-abstractive-question-answering\.venv\Lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Environment ready.


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [3]:
# from datasets import load_dataset

# # load the dataset from huggingface in streaming mode and shuffle it
# wiki_data = load_dataset(
#     'vblagoje/wikipedia_snippets_streamed',
#     split='train',
#     streaming=True
# ).shuffle(seed=960)
# My Snippets Dataset
from datasets import load_dataset

# Load dataset in streaming mode (no full download required)
wiki_data = load_dataset(
    "vblagoje/wikipedia_snippets_streamed",
    split="train",
    streaming=True
)

# Shuffle stream for randomness
wiki_data = wiki_data.shuffle(seed=960)

c:\Users\profe\Documents\Ironhack\lab-abstractive-question-answering\.venv\Lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for vblagoje/wikipedia_snippets_streamed contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/vblagoje/wikipedia_snippets_streamed
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [4]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'wiki_id': 'Q7649565',
 'start_paragraph': 20,
 'start_character': 272,
 'end_paragraph': 24,
 'end_character': 380,
 'article_title': 'Sustainable Agriculture Research and Education',
 'section_title': "2000s & Evaluation of the program's effectiveness",
 'passage_text': "preserving the surrounding prairies. It ran until March 31, 2001.\nIn 2008, SARE celebrated its 20th anniversary. To that date, the program had funded 3,700 projects and was operating with an annual budget of approximately $19 million. Evaluation of the program's effectiveness As of 2008, 64% of farmers who had received SARE grants stated that they had been able to earn increased profits as a result of the funding they received and utilization of sustainable agriculture methods. Additionally, 79% of grantees said that they had experienced a significant improvement in soil quality though the environmentally friendly, sustainable methods that they were"}

In [5]:
# filter only documents with History as section_title - Replace None with your code
history = None

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [6]:
# from tqdm.auto import tqdm  # progress bar

# total_doc_count = 50000

# counter = 0
# docs = []
# # iterate through the dataset and apply our filter
# for d in tqdm(history, total=total_doc_count):
#     # extract the fields we need - article, section, and passage
#     None
#     # increase the counter on every iteration
#     counter += 1

# my code
from datasets import load_dataset
from tqdm.auto import tqdm

# -------------------------------------------------------
# 1. Load the dataset in streaming mode
# -------------------------------------------------------
wiki_data = load_dataset(
    "vblagoje/wikipedia_snippets_streamed",
    split="train",
    streaming=True
)

# Shuffle stream for randomness
wiki_data = wiki_data.shuffle(seed=960, buffer_size=10000)


# -------------------------------------------------------
# 2. Filter only passages whose section title contains
#    the word "History"
# -------------------------------------------------------
history = (
    d for d in wiki_data
    if d.get("section_title") and "History" in d["section_title"]
)


# -------------------------------------------------------
# 3. Iterate and collect 50,000 documents
# -------------------------------------------------------
total_doc_count = 50000
docs = []

for i, d in enumerate(tqdm(history, total=total_doc_count)):

    # Safely extract required fields
    article_title = d.get("article_title", "")
    section_title = d.get("section_title", "")
    passage_text = d.get("passage_text", "")

    # ---------------------------------------------------
    # Recommended structure for retrieval pipelines
    #
    # Embedding models perform better when titles and
    # section context are included in the text being
    # embedded.
    # ---------------------------------------------------
    text_for_embedding = f"{article_title} | {section_title} | {passage_text}"

    docs.append({
        "id": f"doc_{i}",
        "text": text_for_embedding,
        "metadata": {
            "article_title": article_title,
            "section_title": section_title
        }
    })

    # Stop once we reach the target size
    if len(docs) >= total_doc_count:
        break


# -------------------------------------------------------
# 4. Verify dataset
# -------------------------------------------------------
print(f"Collected {len(docs)} documents")
print(docs[0])

100%|█████████▉| 49999/50000 [05:42<00:00, 145.92it/s]

Collected 50000 documents
{'id': 'doc_0', 'text': "Waltham Iron Ore Tramway | History | progressed. They connected to the northern terminus of the Great Northern Railway's Eaton Branch Railway, which tipplers allowed unloading of the narrow gauge ore wagons into standard gauge trucks. In 1913, a new pit was opened to the east of Long Hole, called Bolton's Pit. This was reached by a longer branch of the tramway.\nThe original Long Hole and Green Lane quarries were exhausted by 1920. The company extended the branch to Bolton's pit to the south east to open up the new Reservoir pit, which was on the western shore of Knipton Reservoir. Reservoir pit continued in use until", 'metadata': {'article_title': 'Waltham Iron Ore Tramway', 'section_title': 'History'}}


In [7]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,id,text,metadata
0,doc_0,Waltham Iron Ore Tramway | History | progresse...,"{'article_title': 'Waltham Iron Ore Tramway', ..."
1,doc_1,Wahl (Noble family) | History | of Sweden lost...,"{'article_title': 'Wahl (Noble family)', 'sect..."
2,doc_2,Åråsen Stadion | History | upgrade could not i...,"{'article_title': 'Åråsen Stadion', 'section_t..."
3,doc_3,Air Kufra | Training center & History | the ma...,"{'article_title': 'Air Kufra', 'section_title'..."
4,doc_4,Bani Buhair | Location & History | are seen su...,"{'article_title': 'Bani Buhair', 'section_titl..."


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [8]:
import os
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.environ.get('PINECONE_API_KEY') or 'PINECONE_API_KEY'

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [10]:
index_name = None #give your index a meaningful name

In [11]:
import time

# check if index already exists (it shouldn't if this is first time)
None #initialize the index, and insure the stats are all zeros

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [12]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# load the retriever model from huggingface model hub
retriever = None #load the retriever model from HuggingFace. Use the flax-sentence-embeddings/all_datasets_v3_mpnet-base model
retriever

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [21]:
# # we will use batches of 64
# batch_size = 64

# #You will create embedding for the passage_text variable and be use to include the meta data in each batch
# for i in tqdm(range(0, len(df), batch_size)):
#     # find end of batch
#     None
#     # extract batch
#     None
#     # generate embeddings for batch
#     None
# # check that we have all vectors in index
# index.describe_index_stats()

# My upsert
# -------------------------------------------------------
# 0. Load environment variables
# -------------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


# -------------------------------------------------------
# 1. Initialize Pinecone and connect to index
# -------------------------------------------------------
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "question-answering"   # use your existing index
index = pc.Index(index_name)

print("Connected to Pinecone index:", index_name)


# -------------------------------------------------------
# 2. (Optional) Skip if already indexed
# -------------------------------------------------------
stats = index.describe_index_stats()
existing = stats["namespaces"].get("", {}).get("vector_count", 0)

if existing > 0:
    print(f"Index already contains {existing} vectors — skipping embedding.")
else:

    # -------------------------------------------------------
    # 3. Load embedding model
    # -------------------------------------------------------
    from sentence_transformers import SentenceTransformer
    from tqdm.auto import tqdm
    import torch

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    print(f"Embedding model running on: {device}")


    # -------------------------------------------------------
    # 4. Define batch size
    # -------------------------------------------------------
    batch_size = 64


    # -------------------------------------------------------
    # 5. Generate embeddings and upload to Pinecone
    # -------------------------------------------------------
    for i in tqdm(range(0, len(df), batch_size)):

        i_end = min(i + batch_size, len(df))
        batch = df.iloc[i:i_end]

        texts = batch["text"].tolist()

        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False
        )

        vectors = []

        for j, emb in enumerate(embeddings):

            row = batch.iloc[j]

            vectors.append({
                "id": row["id"],
                "values": emb.tolist(),
                "metadata": {
                    "article_title": row["metadata"]["article_title"],
                    "section_title": row["metadata"]["section_title"],
                    "text": row["text"]
                }
            })

        index.upsert(vectors=vectors)


# -------------------------------------------------------
# 6. Confirm vectors were indexed
# -------------------------------------------------------
print(index.describe_index_stats())

Connected to Pinecone index: question-answering
Index already contains 28992 vectors — skipping embedding.
{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 28992}},
 'total_vector_count': 28992}


# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [22]:
# from transformers import BartTokenizer, BartForConditionalGeneration

# # load bart tokenizer and model from huggingface
# tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
# generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

# Let's do more
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# -------------------------------------------------------
# 1. Detect compute device
# -------------------------------------------------------
# If a GPU exists the generator runs much faster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Generator running on: {device}")


# -------------------------------------------------------
# 2. Load tokenizer
# -------------------------------------------------------
# The tokenizer converts text into tokens understood
# by the BART sequence-to-sequence model
tokenizer = AutoTokenizer.from_pretrained("vblagoje/bart_lfqa")


# -------------------------------------------------------
# 3. Load generator model
# -------------------------------------------------------
# This BART model was fine-tuned for
# Long-Form Question Answering (LFQA)
generator = AutoModelForSeq2SeqLM.from_pretrained(
    "vblagoje/bart_lfqa"
)

# move model to device
generator = generator.to(device)

# set model to evaluation mode (important for inference)
generator.eval()


# -------------------------------------------------------
# 4. Quick sanity test
# -------------------------------------------------------
test_question = "What is a sonic boom?"

inputs = tokenizer(
    f"question: {test_question}",
    return_tensors="pt",
    truncation=True
).to(device)

output = generator.generate(
    **inputs,
    max_length=64
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Generator running on: cpu


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 5158.10it/s]


Sound is a vibration of air. Sound is a vibration of air.


All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [15]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = None
    # search pinecone index for context passage with the answer
    xc = None
    return xc

In [16]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = None 
    # contcatinate the query and context passages
    query = None
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [17]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

In [18]:
from pprint import pprint

In [19]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

print(pc.list_indexes())

{'indexes': [{'dimension': 384,
              'host': 'question-answering-a5fsaua.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'question-answering',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}},
             {'dimension': 384,
              'host': 'qa-index-a5fsaua.svc.aped-4627-b74a.pinecone.io',
              'metric': 'cosine',
              'name': 'qa-index',
              'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
              'status': {'ready': True, 'state': 'Ready'}}]}


In [23]:
stats = index.describe_index_stats()
print(stats)

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 28992}},
 'total_vector_count': 28992}


In [24]:
# # format the query in the form generator expects the input
# query = format_query(query, result["matches"])
# pprint(query)

# My format query
# -------------------------------------------------------
# 0. Load environment variables
# -------------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


# -------------------------------------------------------
# 1. Initialize Pinecone
# -------------------------------------------------------
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "question-answering"
index = pc.Index(index_name)

print("Connected to Pinecone index:", index_name)


# -------------------------------------------------------
# 2. Load Wikipedia dataset (streaming)
# -------------------------------------------------------
from datasets import load_dataset
from tqdm.auto import tqdm

wiki_data = load_dataset(
    "vblagoje/wikipedia_snippets_streamed",
    split="train",
    streaming=True
).shuffle(seed=960, buffer_size=10000)


# -------------------------------------------------------
# 3. Extract 50k "History" passages
# -------------------------------------------------------
docs = []
limit = 5000

for sample in tqdm(wiki_data):

    section = sample.get("section_title", "")

    if "History" in section:

        article_title = sample.get("article_title", "")
        passage_text = sample.get("passage_text", "")

        docs.append({
            "id": f"doc_{len(docs)}",
            "text": f"{article_title} | {section} | {passage_text}",
            "metadata": {
                "article_title": article_title,
                "section_title": section
            }
        })

    if len(docs) >= limit:
        break

print("Collected documents:", len(docs))


# -------------------------------------------------------
# 4. Convert to DataFrame (for batching convenience)
# -------------------------------------------------------
import pandas as pd

df = pd.DataFrame(docs)

print("DataFrame ready:", len(df))


# -------------------------------------------------------
# 5. Load embedding model
# -------------------------------------------------------
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Embedding model running on:", device)


# -------------------------------------------------------
# 6. Batch embedding and Pinecone upload
# -------------------------------------------------------
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):

    i_end = min(i + batch_size, len(df))
    batch = df.iloc[i:i_end]

    texts = batch["text"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False
    )

    vectors = []

    for j, emb in enumerate(embeddings):

        row = batch.iloc[j]

        vectors.append({
            "id": row["id"],
            "values": emb.tolist(),
            "metadata": {
                "article_title": row["metadata"]["article_title"],
                "section_title": row["metadata"]["section_title"],
                "text": row["text"]
            }
        })

    index.upsert(vectors=vectors)


# -------------------------------------------------------
# 7. Confirm indexing succeeded
# -------------------------------------------------------
stats = index.describe_index_stats()

print(stats)

Connected to Pinecone index: question-answering


c:\Users\profe\Documents\Ironhack\lab-abstractive-question-answering\.venv\Lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for vblagoje/wikipedia_snippets_streamed contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/vblagoje/wikipedia_snippets_streamed
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
62593it [00:37, 1681.96it/s]


Collected documents: 5000
DataFrame ready: 5000


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6729.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model running on: cpu


100%|██████████| 79/79 [01:58<00:00,  1.50s/it]

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 28992}},
 'total_vector_count': 28992}


The output looks great. Now let's write a function to generate answers.

In [25]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [26]:
generate_answer(query)

('Electricity was first used in the 19th century. The first electric power '
 'system built was a steam engine.')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [27]:
# query = "How was the first wireless message sent?"
# context = query_pinecone(query, top_k=5)
# query = format_query(query, context["matches"])
# generate_answer(query)

# -------------------------------------------------------
# Ask a question using the full RAG pipeline
# -------------------------------------------------------

question = "How was the first wireless message sent?"


# -------------------------------------------------------
# 1. Embed the question
# -------------------------------------------------------
query_vector = model.encode(question).tolist()


# -------------------------------------------------------
# 2. Retrieve relevant passages from Pinecone
# -------------------------------------------------------
retrieval_result = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)

# Validate result
if retrieval_result is None or "matches" not in retrieval_result:
    raise ValueError("Pinecone query returned no results.")


# -------------------------------------------------------
# 3. Format generator prompt
# -------------------------------------------------------
contexts = []

for match in retrieval_result["matches"]:
    text = match["metadata"].get("text", "")
    if text:
        contexts.append(text)

context_block = " <P> ".join(contexts)

formatted_query = f"question: {question} <P> {context_block}"


# -------------------------------------------------------
# 4. Generate the answer
# -------------------------------------------------------
inputs = tokenizer(
    formatted_query,
    return_tensors="pt",
    truncation=True
).to(device)

output = generator.generate(
    **inputs,
    max_length=256,
    min_length=64,
    num_beams=4,
    early_stopping=True
)

answer = tokenizer.decode(output[0], skip_special_tokens=True)


# -------------------------------------------------------
# 5. Display result
# -------------------------------------------------------
print("\nQuestion:\n", question)
print("\nAnswer:\n", answer)


Question:
 How was the first wireless message sent?

Answer:
 The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s.


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [28]:
# for doc in context["matches"]:
#     print(doc["metadata"]["passage_text"], end='\n---\n')
# -------------------------------------------------------
# Display the passages retrieved from Pinecone
# These were used as context for answer generation
# -------------------------------------------------------

for i, match in enumerate(retrieval_result["matches"], start=1):

    metadata = match.get("metadata", {})
    text = metadata.get("text", "No text found")

    print(f"Context {i}:\n{text}")
    print("\n" + "-" * 60 + "\n")

Context 1:
Unified communications | History | Instead, the handset lived on the network as another computer device.  The transport of audio was therefore no longer a variation in voltages or modulation of frequency such as with the handsets from before, but rather encoding the conversation using a CODEC (G.711 originally) and transporting it with a protocol such as the Real-time Transport Protocol (RTP).  When the handset is just another computer connected to the network, advanced features can be provided by letting computer applications communicate with server computers elsewhere in any number of ways; applications can even be upgraded or freshly installed on the handset.
When

------------------------------------------------------------

Context 2:
WWLP | History | WWLP History WWLP began broadcasting on March 17, 1953 one month before rival WGGB-TV (then known as WHYN-TV). The station aired an analog signal on UHF channel 61 and was an NBC affiliate from the start. At its sign-on, W

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [29]:
# query = "where did COVID-19 originate?"
# context = query_pinecone(query, top_k=3)
# query = format_query(query, context["matches"])
# generate_answer(query)

# -------------------------------------------------------
# Example question outside the dataset scope
# -------------------------------------------------------

question = "Where did COVID-19 originate?"


# -------------------------------------------------------
# 1. Embed the question
# -------------------------------------------------------
query_vector = model.encode(question).tolist()


# -------------------------------------------------------
# 2. Retrieve context from Pinecone
# -------------------------------------------------------
retrieval_result = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

# Ensure retrieval worked
matches = retrieval_result.get("matches", []) if retrieval_result else []

print(f"Retrieved {len(matches)} context passages.\n")


# -------------------------------------------------------
# 3. Format prompt for generator
# -------------------------------------------------------
contexts = []

for match in matches:
    text = match.get("metadata", {}).get("text", "")
    if text:
        contexts.append(text)

context_block = " <P> ".join(contexts)

formatted_query = f"question: {question} <P> {context_block}"


# -------------------------------------------------------
# 4. Generate the answer
# -------------------------------------------------------
inputs = tokenizer(
    formatted_query,
    return_tensors="pt",
    truncation=True
).to(device)

output = generator.generate(
    **inputs,
    max_length=256,
    min_length=64,
    num_beams=4,
    early_stopping=True
)

answer = tokenizer.decode(output[0], skip_special_tokens=True)


# -------------------------------------------------------
# 5. Display result
# -------------------------------------------------------
print("Question:\n", question)
print("\nGenerated Answer:\n", answer)

Retrieved 3 context passages.

Question:
 Where did COVID-19 originate?

Generated Answer:
 COVID-19 is a new strain of the coronaviruses. The coronaviruses that cause coronaviruses are the same ones that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses that cause coronaviruses.


In [31]:
# for doc in context["matches"]:
#     print(doc["metadata"]["passage_text"], end='\n---\n')

# -------------------------------------------------------
# Display retrieved passages used as context
# -------------------------------------------------------

# -------------------------------------------------------
# 1. Define question
# -------------------------------------------------------
question = "What was the War of Currents?"


# -------------------------------------------------------
# 2. Embed the question
# -------------------------------------------------------
query_vector = model.encode(question).tolist()


# -------------------------------------------------------
# 3. Query Pinecone
# -------------------------------------------------------
context = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)


# -------------------------------------------------------
# 4. Safely extract matches
# -------------------------------------------------------
matches = context.get("matches", []) if context else []


# -------------------------------------------------------
# 5. Display retrieved passages
# -------------------------------------------------------
if len(matches) == 0:
    print("No context passages were retrieved.")
else:
    for i, match in enumerate(matches, start=1):

        metadata = match.get("metadata", {})
        passage = metadata.get("text", "No passage text available")

        print(f"Context {i}:\n{passage}")
        print("\n" + "-" * 60 + "\n")

Context 1:
Declaration of war by the United States | History | nations five separate times, each upon prior request by the President of the United States. Four of those five declarations came after hostilities had begun. James Madison reported that in the Federal Convention of 1787, the phrase "make war" was changed to "declare war" in order to leave to the Executive the power to repel sudden attacks but not to commence war without the explicit approval of Congress. Debate continues as to the legal extent of the President's authority in this regard. Public opposition to American involvement in foreign wars, particularly during the 1930s, was expressed as support for a

------------------------------------------------------------

Context 2:
Troop engagements of the American Civil War, 1861 | History | forces remaining in the territory and the Apache tribes.

------------------------------------------------------------

Context 3:
Coal Wars | History | of 1912, and The Battle of Evarts,

Let’s finish with a final few questions.

In [33]:
# def new_func():
#     query = "what was the war of currents?"
#     context = query_pinecone(query, top_k=5)
#     query = format_query(query, context["matches"])
#     generate_answer(query)

# new_func()

# -------------------------------------------------------
# Ask a question safely using the RAG pipeline
# -------------------------------------------------------

question = "What was the War of Currents?"

# 1. Retrieve context
context = query_pinecone(question, top_k=5)

matches = context.get("matches", []) if context else []

print(f"Retrieved {len(matches)} context passages.\n")


# -------------------------------------------------------
# 2. Format query safely
# -------------------------------------------------------
contexts = []

for match in matches:
    text = match.get("metadata", {}).get("text", "")
    if text:
        contexts.append(text)

# Fallback if no context retrieved
if len(contexts) == 0:
    print("⚠️ No context found — using question only.\n")
    formatted_query = f"question: {question}"
else:
    context_block = " <P> ".join(contexts)
    formatted_query = f"question: {question} <P> {context_block}"


# Ensure correct type (critical fix)
formatted_query = str(formatted_query)


# -------------------------------------------------------
# 3. Generate answer
# -------------------------------------------------------
answer = generate_answer(formatted_query)


# -------------------------------------------------------
# 4. Display result
# -------------------------------------------------------
print("Question:\n", question)
print("\nAnswer:\n", answer)

Retrieved 0 context passages.

⚠️ No context found — using question only.

('The War of Currents was a conflict between the US Navy and the Royal Navy. '
 'It was fought in the mid-1930s.')
Question:
 What was the War of Currents?

Answer:
 None


In [36]:
# query = "who was the first person on the moon?"
# context = query_pinecone(query, top_k=10)
# query = format_query(query, context["matches"])
# generate_answer(query)

# -------------------------------------------------------
# Ask a question using the RAG pipeline (safe version)
# -------------------------------------------------------

question = "Who was the first person on the moon?"

# 1. Retrieve context
context = query_pinecone(question, top_k=10)

matches = context.get("matches", []) if context else []

print(f"Retrieved {len(matches)} context passages.\n")


# -------------------------------------------------------
# 2. Build formatted query (SAFE)
# -------------------------------------------------------
contexts = []

for match in matches:
    text = match.get("metadata", {}).get("text", "")
    if isinstance(text, str) and text.strip():
        contexts.append(text)

# fallback if retrieval failed
if len(contexts) == 0:
    print("⚠️ No context retrieved — using question only.\n")
    formatted_query = f"question: {question}"
else:
    context_block = " <P> ".join(contexts)
    formatted_query = f"question: {question} <P> {context_block}"

# 🔴 critical: guarantee correct type
formatted_query = str(formatted_query)


# -------------------------------------------------------
# 3. Generate answer
# -------------------------------------------------------
answer = generate_answer(formatted_query)


# -------------------------------------------------------
# 4. Display result
# -------------------------------------------------------
print("Question:\n", question)
print("\nAnswer:\n", answer)

Retrieved 0 context passages.

⚠️ No context retrieved — using question only.

('The first man to walk on the moon was Neil Armstrong. He was born in 1946. '
 'Edit:')
Question:
 Who was the first person on the moon?

Answer:
 None


In [38]:
# query = "what was NASAs most expensive project?"
# context = query_pinecone(query, top_k=3)
# query = format_query(query, context["matches"])
# generate_answer(query)

# -------------------------------------------------------
# Ask a question using the RAG pipeline (FIXED)
# -------------------------------------------------------

question = "What was NASA's most expensive project?"

# 1. Retrieve context
context = query_pinecone(question, top_k=3)

matches = context.get("matches", []) if context else []

print(f"Retrieved {len(matches)} context passages.\n")


# -------------------------------------------------------
# 2. Build prompt manually (SAFE)
# -------------------------------------------------------
contexts = []

for match in matches:
    text = match.get("metadata", {}).get("text", "")
    if isinstance(text, str) and text.strip():
        contexts.append(text)

if len(contexts) == 0:
    print("⚠️ No context retrieved — using question only.\n")
    formatted_query = f"question: {question}"
else:
    context_block = " <P> ".join(contexts)
    formatted_query = f"question: {question} <P> {context_block}"

# 🔴 guarantee correct type
formatted_query = str(formatted_query)


# -------------------------------------------------------
# 3. Generate answer
# -------------------------------------------------------
answer = generate_answer(formatted_query)


# -------------------------------------------------------
# 4. Display result
# -------------------------------------------------------
print("Question:\n", question)
print("\nAnswer:\n", answer)

Retrieved 0 context passages.

⚠️ No context retrieved — using question only.

'The Apollo 11 mission was the most expensive. It cost about $2.5 billion USD.'
Question:
 What was NASA's most expensive project?

Answer:
 None


As we can see, the model can generate some decent answers.

#### Add a few more questions

In [40]:
# -------------------------------------------------------
# Ask a question about New Order (FIXED)
# -------------------------------------------------------

question = "How did the band New Order form after the breakup of Joy Division?"

# 1. Query Pinecone for relevant context
context = query_pinecone(question, top_k=5)

# Safely extract matches
matches = context.get("matches", []) if context else []

print(f"Retrieved {len(matches)} context passages.\n")


# -------------------------------------------------------
# 2. Build formatted query (SAFE)
# -------------------------------------------------------
contexts = []

for match in matches:
    text = match.get("metadata", {}).get("text", "")
    if isinstance(text, str) and text.strip():
        contexts.append(text)

if len(contexts) == 0:
    print("⚠️ No context retrieved — using question only.\n")
    formatted_query = f"question: {question}"
else:
    context_block = " <P> ".join(contexts)
    formatted_query = f"question: {question} <P> {context_block}"

# 🔴 ensure correct type
formatted_query = str(formatted_query)


# -------------------------------------------------------
# 3. Generate the answer
# -------------------------------------------------------
answer = generate_answer(formatted_query)


# -------------------------------------------------------
# 4. Display result
# -------------------------------------------------------
print("Question:\n", question)
print("\nAnswer:\n", answer)

Retrieved 0 context passages.

⚠️ No context retrieved — using question only.

('New Order was formed by members of Joy Division. The band was formed in '
 '1981, after the breakup of Joy Division.')
Question:
 How did the band New Order form after the breakup of Joy Division?

Answer:
 None


*** all done